# OPSD v14 — KL Distillation Training
### Sampled-token KL divergence · Starts from SFT checkpoint · Saves to opsd_kl_adapter

---


## 00 · Smart Install

> Run once → Restart Kernel.

In [1]:
import subprocess, sys
pip = [sys.executable, '-m', 'pip']

def run(cmd, label):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {'✅' if r.returncode==0 else '❌'} {label}")
    if r.returncode != 0: print(f'     {r.stderr[-300:].strip()}')

def installed_ver(pkg):
    r = subprocess.run(pip+['show',pkg], capture_output=True, text=True)
    for ln in r.stdout.splitlines():
        if ln.lower().startswith('version:'): return ln.split(':',1)[1].strip()
    return None

def cuda_torch_ok():
    try:
        import torch
        return torch.__version__.startswith('2.3') and ('+cu' in torch.__version__ or torch.cuda.is_available())
    except: return False

print('Step 1 — PyTorch CUDA...')
if cuda_torch_ok():
    print('  ✅ torch 2.3+CUDA already present — skipped')
else:
    subprocess.run(pip+['uninstall','-y','torch','torchvision','torchaudio'], capture_output=True)
    run(pip+['install','-q','torch==2.3.1','torchvision==0.18.1','torchaudio==2.3.1',
             '--index-url','https://download.pytorch.org/whl/cu121'], 'torch 2.3.1+cu121')

print('\nStep 2 — ML stack...')
pkgs = [
    ('transformers==4.47.1','transformers','4.47'),
    ('peft==0.13.2','peft','0.13'),
    ('accelerate==1.2.1','accelerate','1.2'),
    ('datasets==2.21.0','datasets','2.21'),
    ('trl==0.12.2','trl','0.12'),
    ('bitsandbytes==0.45.3','bitsandbytes','0.45'),
    ('sentencepiece','sentencepiece',None),
    ('protobuf','protobuf',None),
    ('matplotlib','matplotlib',None),
    ('tqdm','tqdm',None),
]
for spec, name, prefix in pkgs:
    v = installed_ver(name)
    if v and (prefix is None or v.startswith(prefix)):
        print(f'  ✅ {name:28s} {v} — skipped')
    else:
        print(f'  ⬇  {name:28s} installing...')
        run(pip+['install','-q',spec], name)

print('\n'+'='*55)
print('  ✅ Done! RESTART KERNEL NOW  (Kernel → Restart)')
print('='*55)


Step 1 — PyTorch CUDA...
  ✅ torch 2.3+CUDA already present — skipped

Step 2 — ML stack...
  ✅ transformers                 4.47.1 — skipped
  ✅ peft                         0.13.2 — skipped
  ✅ accelerate                   1.2.1 — skipped
  ✅ datasets                     2.21.0 — skipped
  ✅ trl                          0.12.2 — skipped
  ✅ bitsandbytes                 0.45.3 — skipped
  ✅ sentencepiece                0.2.1 — skipped
  ✅ protobuf                     5.29.5 — skipped
  ✅ matplotlib                   3.10.8 — skipped
  ✅ tqdm                         4.67.1 — skipped

  ✅ Done! RESTART KERNEL NOW  (Kernel → Restart)


---
## 01 · Verify GPU

In [1]:
import torch, importlib
print('GPU'); print('-'*40)
assert torch.cuda.is_available(), 'No CUDA GPU! Re-run cell 00 and restart kernel.'
print(f'  ✅ {torch.cuda.get_device_name(0)}')
print(f'  ✅ VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'  ✅ CUDA : {torch.version.cuda}')
checks = [('torch','2.3'),('transformers','4.47'),('peft','0.13'),
          ('accelerate','1.2'),('datasets','2.21'),('trl','0.12'),('bitsandbytes','0.45')]
print('\nPACKAGES')
for pkg,exp in checks:
    v = importlib.import_module(pkg).__version__
    print(f"  {'✅' if v.startswith(exp) else '⚠️ '} {pkg:15s} {v}")
print('\n✅ Ready!')


GPU
----------------------------------------
  ✅ NVIDIA GeForce RTX 4070 Laptop GPU
  ✅ VRAM : 8.6 GB
  ✅ CUDA : 12.1

PACKAGES
  ✅ torch           2.3.1+cu121
  ✅ transformers    4.47.1
  ✅ peft            0.13.2
  ✅ accelerate      1.2.1
  ✅ datasets        2.21.0
  ✅ trl             0.12.2
  ✅ bitsandbytes    0.45.3

✅ Ready!


## 02 · Imports & Config

In [2]:
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
import copy, math, gc, warnings, re, os, time
from pathlib import Path
from datetime import timedelta
from torch.cuda.amp import autocast
from tqdm.notebook import tqdm as tqdm_nb
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

DEVICE = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark        = True

# ══════════════════════════════════════════════════
#  CONFIGURATION — v14 fixed
# ══════════════════════════════════════════════════
MODEL_NAME    = 'Qwen/Qwen2.5-3B-Instruct'
DATASET_SIZE  = 1000
EPOCHS        = 1
LR            = 2e-5
GRAD_ACCUM    = 4
TEMPERATURE   = 1.0
LORA_R        = 64
LORA_ALPHA    = 128
MAX_GEN_LEN   = 200
GRPO_G        = 4
BATCH_SIZE    = 4

CKPT_DIR      = Path('./checkpoints'); CKPT_DIR.mkdir(exist_ok=True)
SFT_CKPT      = CKPT_DIR / 'sft_adapter'       # existing — load as baseline
OPSD_ABS_CKPT = CKPT_DIR / 'opsd_adapter'      # old abs loss — keep for comparison
OPSD_KL_CKPT  = CKPT_DIR / 'opsd_kl_adapter'   # KL loss version
JSD_CKPT      = CKPT_DIR / 'jsd_adapter'        # FIX: JSD training (was wrongly grpo_adapter)
GRPO_CKPT     = CKPT_DIR / 'grpo_adapter'       # kept for reference if it exists from old runs
EVAL_RESULTS_FILE = CKPT_DIR / 'eval_results_v14.json'

def fmt_time(s): return str(timedelta(seconds=int(s)))

print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Batch  : {BATCH_SIZE} samples at once')
print(f'Data   : {DATASET_SIZE} samples')
print(f'Ckpts  : {CKPT_DIR.resolve()}')
print()
print('Checkpoint status:')
for name, path in [('SFT', SFT_CKPT), ('OPSD-abs', OPSD_ABS_CKPT),
                   ('OPSD-KL', OPSD_KL_CKPT), ('JSD', JSD_CKPT), ('GRPO (old)', GRPO_CKPT)]:
    print(f"  {'✅' if path.exists() else '❌'} {name:18s} {path}")


GPU    : NVIDIA GeForce RTX 4070 Laptop GPU
VRAM   : 8.6 GB
Batch  : 4 samples at once
Data   : 1000 samples
Ckpts  : D:\Project\model distlation\checkpoints

Checkpoint status:
  ✅ SFT                checkpoints\sft_adapter
  ✅ OPSD-abs           checkpoints\opsd_adapter
  ✅ OPSD-KL            checkpoints\opsd_kl_adapter
  ✅ JSD                checkpoints\jsd_adapter
  ✅ GRPO (old)         checkpoints\grpo_adapter


## 03 · Load Model

> Checks HuggingFace cache first — skips 6 GB download if already on disk.

In [3]:
def is_model_cached(model_name):
    folder = 'models--' + model_name.replace('/', '--')
    hf_home = Path(os.environ.get('HUGGINGFACE_HUB_CACHE') or
                   os.environ.get('HF_HOME', Path.home()/'.cache'/'huggingface'/'hub'))
    snaps = hf_home / folder / 'snapshots'
    if snaps.exists():
        files = list(snaps.rglob('*.safetensors'))
        if files: return True, sum(f.stat().st_size for f in files)/1e9
    return False, 0.0

cached, gb = is_model_cached(MODEL_NAME)
if cached:
    print(f'✅ Model in cache ({gb:.1f} GB) — loading from disk, no download needed!')
else:
    print('Not in cache — will download ~6 GB (first time only)...')

# ── VRAM-safe model loading ───────────────────────────────────────────────
# Step 1: check free VRAM before loading
free_vram = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_allocated()) / 1e9
print(f'Free VRAM before load: {free_vram:.1f} GB')
if free_vram < 2.0:
    print('⚠️  Low VRAM — run: gc.collect(); torch.cuda.empty_cache() then re-run this cell')

# Step 2: flush any leftover tensors from a previous failed load
import gc
if 'base_model' in dir(): del base_model
gc.collect(); torch.cuda.empty_cache()

# Step 3: configure 4-bit quant with CPU offload allowed for lm_head only
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,  # allows lm_head to stay fp32 on CPU if needed
)

# Step 4: build a device_map that keeps all transformer layers on GPU
# but allows the embedding/lm_head to offload if VRAM is tight
try:
    t0 = time.time()
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_cfg,
        device_map='auto', trust_remote_code=True, attn_implementation='eager',
    )
    print(f'✅ Loaded with device_map=auto in {time.time()-t0:.0f}s')
except ValueError as e:
    if 'CPU or the disk' in str(e):
        print('⚠️  device_map=auto tried to spill to CPU — switching to cuda:0 only...')
        gc.collect(); torch.cuda.empty_cache()
        t0 = time.time()
        # Force everything onto GPU — will OOM if truly not enough VRAM
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_cfg,
            device_map={'': 0}, trust_remote_code=True, attn_implementation='eager',
        )
        print(f'✅ Loaded with device_map=cuda:0 in {time.time()-t0:.0f}s')
    else:
        raise

base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False
# ─────────────────────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'   # required for batched generation
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

print(f'✅ Loaded in {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')


✅ Model in cache (6.2 GB) — loading from disk, no download needed!
Free VRAM before load: 8.6 GB


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Loaded with device_map=auto in 14s
✅ Loaded in 16s | VRAM: 2.1 GB


In [4]:
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
student_model = get_peft_model(base_model, lora_config)
student_model.train()
student_model.print_trainable_parameters()

teacher_model = copy.deepcopy(student_model)
teacher_model.eval()
for p in teacher_model.parameters(): p.requires_grad = False
print(f'✅ Student + Teacher ready | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')


trainable params: 119,734,272 || all params: 3,205,672,960 || trainable%: 3.7351
✅ Student + Teacher ready | VRAM: 5.1 GB


## 04 · Dataset

In [ ]:
raw  = load_dataset('gsm8k', 'main', split=f'train[:{DATASET_SIZE}]')
data = [(s['question'], s['answer']) for s in raw]
print(f'✅ {len(data)} GSM8K samples')
# Create batches
batches = [data[i:i+BATCH_SIZE] for i in range(0, len(data), BATCH_SIZE)]
print(f'   {len(batches)} batches of size {BATCH_SIZE}')


## 05 · SFT Baseline — Batched

> Processes multiple samples per forward pass. Saves checkpoint on completion.
> Run **Cell 05b** instead if you already have a saved checkpoint.

In [ ]:
# ── Cell 05 · Load SFT checkpoint as baseline for KL-OPSD ───────────────────
# We ALWAYS start from SFT for the KL experiment (clean baseline).
# Do NOT continue from old opsd_adapter — different objective.
import json as _j
from peft import PeftModel

if SFT_CKPT.exists():
    print('=' * 55)
    print('  ✅ SFT CHECKPOINT FOUND')
    print(f'  📁 {SFT_CKPT}')
    _lf = CKPT_DIR / 'sft_losses.json'
    sft_losses = _j.loads(_lf.read_text()) if _lf.exists() else []
    if sft_losses:
        print(f'  📉 Last loss : {sft_losses[-1]:.4f}')
        print(f'  📊 Steps done: {len(sft_losses)}')
    print('  Loading SFT checkpoint as KL-OPSD starting point...')
    print('=' * 55)
    student_model = PeftModel.from_pretrained(base_model, str(SFT_CKPT))
    student_model.train()
    student_model.enable_input_require_grads()
    print('  ✅ SFT loaded. Proceed to Cell 06 (KL-OPSD).')
    print('=' * 55)
    

else:
    print('❌ SFT checkpoint not found at:', SFT_CKPT)
    print('   Run the original notebook first to create the SFT checkpoint.')
    print('   Or reduce DATASET_SIZE to 300 and run SFT from scratch below.')
    print()
    print('   Running SFT from scratch (~13 min)...')
    student_model.enable_input_require_grads()

    def sft_train(model, data, epochs=EPOCHS, lr=LR, grad_accum=GRAD_ACCUM):
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
        model.train()
        losses, times = [], []
        optimizer.zero_grad()
        total = len(data) * epochs
        bar = tqdm_nb(total=total, desc='SFT')
        train_start = time.time()
        for epoch in range(epochs):
            for step, (x, y) in enumerate(data):
                t0 = time.time()
                prompt = f'Problem:\n{x}\nSolution:\n{y}'
                enc = tokenizer(prompt, return_tensors='pt', truncation=True,
                                max_length=400, padding=True).to(DEVICE)
                with autocast(dtype=torch.bfloat16):
                    out  = model(**enc, labels=enc['input_ids'])
                    loss = out.loss / grad_accum
                loss.backward()
                losses.append(loss.item() * grad_accum)
                if (step + 1) % grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(
                        filter(lambda p: p.requires_grad, model.parameters()), 1.0)
                    optimizer.step(); optimizer.zero_grad()
                del enc, out, loss
                gc.collect(); torch.cuda.empty_cache()
                times.append(time.time() - t0)
                avg_t = sum(times[-20:]) / len(times[-20:])
                done  = epoch * len(data) + step + 1
                eta   = avg_t * (total - done)
                vram  = torch.cuda.memory_allocated() / 1e9
                bar.update(1)
                bar.set_postfix({'loss': f'{losses[-1]:.4f}',
                                 'ETA': fmt_time(eta), 'VRAM': f'{vram:.1f}GB'})
                print(f'  [{done:3d}/{total}] loss={losses[-1]:.4f} '
                      f'elapsed={fmt_time(time.time()-train_start)} '
                      f'ETA={fmt_time(eta)} VRAM={vram:.1f}GB', flush=True)
        bar.close()
        return losses

    print('Starting SFT...')
    t0 = time.time()
    sft_losses = sft_train(student_model, data)
    print(f'\n✅ SFT done in {fmt_time(time.time()-t0)} | loss={sft_losses[-1]:.4f}')
    student_model.save_pretrained(str(SFT_CKPT))
    tokenizer.save_pretrained(str(SFT_CKPT))
    import json as _j2
    (CKPT_DIR / 'sft_losses.json').write_text(_j2.dumps(sft_losses))
    print(f'✅ Saved → {SFT_CKPT}')
    print('▶  Proceed to Cell 06 (KL-OPSD).')


## 05b · Load SFT Checkpoint *(skip 05 if already trained)*

In [ ]:
import json as _j
from peft import PeftModel
if not SFT_CKPT.exists():
    print('❌ No checkpoint — run Cell 05 first')
else:
    t0 = time.time()
    student_model = PeftModel.from_pretrained(base_model, str(SFT_CKPT))
    student_model.train(); student_model.enable_input_require_grads()
    sft_losses = _j.loads((CKPT_DIR/'sft_losses.json').read_text()) if (CKPT_DIR/'sft_losses.json').exists() else []
    print(f'✅ SFT loaded in {time.time()-t0:.1f}s | {len(sft_losses)} steps | loss={sft_losses[-1]:.4f}')


## 06 · OPSD-KL — Sampled-Token KL Training

**Loss change from v12:**
-  Old: `abs(student_logprob - teacher_logprob)` — wrong gradient direction
-  New: `(teacher_logprob - student_logprob)` — sampled-token KL

**Why this is correct:**
- Gradient wrt student: `-1` → always pushes student toward teacher
- If student too low → increase it
- If student too high → decrease it
- Matches paper Eq. 9 directional objective

**Monitoring:** Watch `gap` shrinking → alignment improving.

> Saves to `opsd_kl_adapter` — separate from old `opsd_adapter`

In [ ]:
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9
print(f'Free VRAM: {free:.1f} GB')
if free < 1.5:
    print('⚠️  WARNING: Very low VRAM — reduce BATCH_SIZE to 2 in Cell 02')
else:
    print('✅ VRAM OK')


In [ ]:
# ── Cell 06 · OPSD-KL — Fully Resumable + Multi-Checkpoint (Stable) ─────────

import json as _j
import gc, torch
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM

RESUME_META = CKPT_DIR / "opsd_kl_state.json"
LOSS_FILE   = CKPT_DIR / "opsd_kl_losses.json"

SAVE_EVERY = 10   # Save every N batches

# ─────────────────────────────────────────────────────────────────────────────
# 1️⃣ LOAD OR RESUME STUDENT
# ─────────────────────────────────────────────────────────────────────────────

start_batch = 0
opsd_kl_losses = []

if OPSD_KL_CKPT.exists():
    print("🔁 Existing OPSD-KL checkpoint detected")

    student_model = PeftModel.from_pretrained(base_model, str(OPSD_KL_CKPT))
    student_model.train()
    student_model.enable_input_require_grads()

    # Re-enable LoRA params
    for name, param in student_model.named_parameters():
        param.requires_grad = ('lora_' in name)

    trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
    print(f"Trainable params after resume: {trainable:,}")

    if RESUME_META.exists():
        meta = _j.loads(RESUME_META.read_text())
        start_batch = meta.get("last_batch", 0)
        print(f"Resuming from batch {start_batch}")

    if LOSS_FILE.exists():
        opsd_kl_losses = _j.loads(LOSS_FILE.read_text())

else:
    print("🚀 No checkpoint found — starting fresh")

    for name, param in student_model.named_parameters():
        param.requires_grad = ('lora_' in name)

    trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,}")

# ─────────────────────────────────────────────────────────────────────────────
# 2️⃣ BUILD TEACHER (CPU ONLY)
# ─────────────────────────────────────────────────────────────────────────────

gc.collect()
torch.cuda.empty_cache()

teacher_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=None,
    trust_remote_code=True
).to("cpu")

teacher_model = PeftModel.from_pretrained(
    teacher_base,
    str(SFT_CKPT),
    device_map=None
)

teacher_model.to("cpu")
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

print("✅ Teacher built fully on CPU")

# ─────────────────────────────────────────────────────────────────────────────
# 3️⃣ HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def build_prompts(batch_xy):
    s_prompts, t_prompts = [], []
    for x, y_star in batch_xy:
        s_prompts.append(
            f"Problem:\n{x}\n"
            f"Think step by step, give final answer after ####.\nSolution:\n"
        )
        t_prompts.append(
            f"Problem:\n{x}\nReference answer:\n{y_star}\n"
            f"Give step-by-step solution ending with ####.\nSolution:\n"
        )
    return s_prompts, t_prompts


def tokenize_batch(prompts, device, max_len=160):
    return tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len
    ).to(device)


@torch.no_grad()
def generate_batch(model, enc):
    return model.generate(
        **enc,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )


def compute_log_probs(model, input_ids, prefix_lens, device, requires_grad=False):

    ctx = torch.enable_grad() if requires_grad else torch.no_grad()

    with ctx:
        out = model(input_ids=input_ids, use_cache=False)
        logits = out.logits.float()

    log_probs_list = []

    for i, plen in enumerate(prefix_lens):
        gen_logits = logits[i, plen-1:-1]
        gen_tokens = input_ids[i, plen:]

        lp = F.log_softmax(gen_logits, dim=-1)
        tok_lp = lp.gather(1, gen_tokens.unsqueeze(1)).squeeze(1)

        if tok_lp.numel() > 0:
            valid = tok_lp.mean()
        else:
            valid = torch.tensor(0.0, device=device)

        log_probs_list.append(valid)

    return log_probs_list

# ─────────────────────────────────────────────────────────────────────────────
# 4️⃣ TRAINING LOOP (RESUMABLE)
# ─────────────────────────────────────────────────────────────────────────────

def opsd_kl_train(student, teacher, data):

    student.train()
    teacher.eval()

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, student.parameters()),
        lr=LR
    )

    batches = [data[i:i+BATCH_SIZE] for i in range(0, len(data), BATCH_SIZE)]
    total_batches = len(batches)

    print(f"Total batches: {total_batches}")
    print(f"Starting from batch: {start_batch}")

    bar = tqdm_nb(total=total_batches, desc="OPSD-KL")
    bar.update(start_batch)

    for b_idx in range(start_batch, total_batches):

        batch = batches[b_idx]
        s_prompts, t_prompts = build_prompts(batch)

        # Student (CUDA)
        s_enc = tokenize_batch(s_prompts, DEVICE)
        s_prefix = s_enc["attention_mask"].sum(dim=1).tolist()

        student.eval()
        s_gen = generate_batch(student, s_enc)
        student.train()

        # Teacher (CPU)
        t_enc = tokenize_batch(t_prompts, "cpu")
        t_prefix = t_enc["attention_mask"].sum(dim=1).tolist()
        t_gen = generate_batch(teacher, t_enc)

        s_lp = compute_log_probs(student, s_gen, s_prefix, DEVICE, requires_grad=True)
        t_lp = compute_log_probs(teacher, t_gen, t_prefix, "cpu", requires_grad=False)
        t_lp = [x.detach().to(DEVICE) for x in t_lp]

        # KL ≈ log p_T - log p_S
        kl_terms = [t - s for s, t in zip(s_lp, t_lp)]
        loss = torch.stack(kl_terms).mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, student.parameters()), 1.0
        )
        optimizer.step()
        optimizer.zero_grad()

        opsd_kl_losses.append(loss.item())

        # Periodic save
        if (b_idx + 1) % SAVE_EVERY == 0:
            print(f"\n💾 Saving checkpoint at batch {b_idx+1}")

            student.save_pretrained(str(OPSD_KL_CKPT))
            tokenizer.save_pretrained(str(OPSD_KL_CKPT))

            LOSS_FILE.write_text(_j.dumps(opsd_kl_losses))
            RESUME_META.write_text(_j.dumps({"last_batch": b_idx+1}))

        bar.update(1)
        bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "VRAM": f"{torch.cuda.memory_allocated()/1e9:.1f}GB"
        })

        del s_enc, t_enc, s_gen, t_gen, s_lp, t_lp
        gc.collect()
        torch.cuda.empty_cache()

    bar.close()

    print("\n✅ Training complete — final save")

    student.save_pretrained(str(OPSD_KL_CKPT))
    tokenizer.save_pretrained(str(OPSD_KL_CKPT))
    LOSS_FILE.write_text(_j.dumps(opsd_kl_losses))
    RESUME_META.write_text(_j.dumps({"last_batch": total_batches}))

    return opsd_kl_losses


# ─────────────────────────────────────────────────────────────────────────────

print("Starting OPSD-KL training...")
opsd_kl_losses = opsd_kl_train(student_model, teacher_model, data)

print(f"\n✅ OPSD-KL done | Final loss: {opsd_kl_losses[-1]:.4f}")

In [ ]:
# ---- MANUAL EMERGENCY SAVE ----
student_model.save_pretrained(str(OPSD_KL_CKPT))
tokenizer.save_pretrained(str(OPSD_KL_CKPT))

# Save current loss history if exists
import json as _j
from pathlib import Path

LOSS_FILE = CKPT_DIR / "opsd_kl_losses.json"
STATE_FILE = CKPT_DIR / "opsd_kl_state.json"

try:
    LOSS_FILE.write_text(_j.dumps(opsd_kl_losses))
except:
    print("Loss history not available")

# We don't know exact last batch, so estimate from loss length
try:
    STATE_FILE.write_text(_j.dumps({"last_batch": len(opsd_kl_losses)}))
except:
    print("Batch count not available")

print("✅ Manual checkpoint + state saved successfully.")

## 06b · Load OPSD-KL Checkpoint *(skip 06 if already trained)*

**FIX:** Corrected path from `opsd_kl` → `opsd_kl_adapter`

In [ ]:
# Cell 06b — uses OPSD_KL_CKPT defined in Cell 02 (no redefinition needed)


In [ ]:
import json as _j
from peft import PeftModel

if not OPSD_KL_CKPT.exists():
    print('❌ No OPSD-KL checkpoint — run Cell 06 first')
else:
    t0 = time.time()
    student_model = PeftModel.from_pretrained(base_model, str(OPSD_KL_CKPT))
    student_model.train(); student_model.enable_input_require_grads()
    opsd_kl_losses = _j.loads((CKPT_DIR/'opsd_kl_losses.json').read_text()) \
                     if (CKPT_DIR/'opsd_kl_losses.json').exists() else []
    if opsd_kl_losses:
        print(f'✅ OPSD-KL loaded in {time.time()-t0:.1f}s | {len(opsd_kl_losses)} steps | loss={opsd_kl_losses[-1]:.4f}')
    else:
        print(f'✅ OPSD-KL loaded in {time.time()-t0:.1f}s')


## 07 · JSD Training

**FIX from v13:** Was mislabeled as GRPO and saving to `grpo_adapter`.
Now correctly named JSD — saves to `jsd_adapter`.
Training logic is **unchanged**.

In [ ]:
def extract_answer(text):
    if '####' in text: return text.split('####')[-1].strip()
    nums = re.findall(r'-?\d+\.?\d*', text)
    return nums[-1] if nums else ''


def opsd_train(model, data, epochs=EPOCHS,
               lr=LR, grad_accum=GRAD_ACCUM):

    # ── memory cleanup ───────────────────────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()

    model.enable_input_require_grads(); model.train()
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    if len(trainable_params) == 0:
        for p in model.parameters():
            if p.dtype in (torch.float32, torch.float16, torch.bfloat16):
                p.requires_grad_(True)
        trainable_params = [p for p in model.parameters() if p.requires_grad]

    print(f"Trainable parameters: {sum(p.numel() for p in trainable_params):,}")
    optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    optimizer.zero_grad()

    # shuffle data each epoch
    import random
    total_batches = len(data) * epochs
    losses, times = [], []
    total_toks = 0
    train_start = time.time()
    bar = tqdm_nb(total=total_batches, desc='OPSD steps')

    MAX_PROMPT = 256   # max tokens for prompt
    MAX_GEN    = 100   # max new tokens to generate
    MAX_SEQ    = MAX_PROMPT + MAX_GEN

    for epoch in range(epochs):
        print(f'\n── Epoch {epoch+1}/{epochs} ──────────────────────', flush=True)
        epoch_data = data.copy(); random.shuffle(epoch_data)

        for step, (x, y_star) in enumerate(epoch_data):

            t0 = time.time()
            gt = extract_answer(y_star)
            if not gt:
                bar.update(1); continue

            # ── OPSD: two prompts, same model ────────────────────────────
            # Student sees problem only
            student_prompt = (
                f'Problem:\n{x}\n'
                f'Think step by step, answer after ####.\nSolution:\n'
            )
            # Teacher sees problem + ground truth answer
            teacher_prompt = (
                f'Problem:\n{x}\n'
                f'Reference answer: {gt}\n'
                f'Now solve step by step, answer after ####.\nSolution:\n'
            )

            # ── Step 1: generate rollout with STUDENT prompt ─────────────
            student_enc = tokenizer(
                student_prompt, return_tensors='pt',
                truncation=True, max_length=MAX_PROMPT
            ).to(DEVICE)
            prefix_len = student_enc['input_ids'].shape[1]

            model.eval()
            with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
                try:
                    gen_ids = model.generate(
                        **student_enc,
                        max_new_tokens=MAX_GEN,
                        do_sample=True,
                        temperature=TEMPERATURE,
                        repetition_penalty=1.1,
                        use_cache=True,
                        pad_token_id=tokenizer.pad_token_id)
                except RuntimeError:
                    gc.collect(); torch.cuda.empty_cache()
                    gen_ids = model.generate(
                        **student_enc,
                        max_new_tokens=MAX_GEN,
                        do_sample=False,
                        repetition_penalty=1.1,
                        use_cache=True,
                        pad_token_id=tokenizer.pad_token_id)

            gen_ids = gen_ids[:, :MAX_SEQ]
            # only the generated portion (no prompt)
            gen_only = gen_ids[0, prefix_len:].cpu()
            total_toks += gen_only.numel()

            del student_enc
            gc.collect(); torch.cuda.empty_cache()
            model.train()

            if gen_only.numel() == 0:
                bar.update(1); continue

            # ── Step 2: build teacher input = teacher_prompt + gen_only ──
            # Teacher forward: problem + gt_answer as context, then
            # runs over the SAME generated tokens the student produced
            teacher_enc = tokenizer(
                teacher_prompt, return_tensors='pt',
                truncation=True, max_length=MAX_PROMPT
            ).to(DEVICE)
            teacher_prefix_len = teacher_enc['input_ids'].shape[1]

            gen_only_gpu = gen_only.to(DEVICE).unsqueeze(0)  # [1, gen_len]

            # Concatenate teacher prompt + student's generated tokens
            teacher_ids  = torch.cat([teacher_enc['input_ids'],  gen_only_gpu], dim=1)
            teacher_attn = torch.ones_like(teacher_ids)
            del teacher_enc
            gc.collect(); torch.cuda.empty_cache()

            # ── Step 3: teacher forward pass (no grad) ───────────────────
            with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
                t_out = model(input_ids=teacher_ids, attention_mask=teacher_attn,
                              use_cache=False, return_dict=True)
            # logits over generated tokens only
            t_logits = t_out.logits[0, teacher_prefix_len-1:-1].float()  # [gen_len, vocab]
            del t_out
            gc.collect(); torch.cuda.empty_cache()

            # ── Step 4: student forward pass (WITH grad) ─────────────────
            student_ids  = gen_ids.to(DEVICE)                  # [1, prompt+gen]
            student_attn = torch.ones_like(student_ids)

            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                s_out = model(input_ids=student_ids, attention_mask=student_attn,
                              use_cache=False, return_dict=True)
            s_logits = s_out.logits[0, prefix_len-1:-1].float()  # [gen_len, vocab]
            del s_out, student_ids, student_attn, gen_ids
            gc.collect(); torch.cuda.empty_cache()

            # ── Step 5: JSD loss between teacher and student ─────────────
            # Jensen-Shannon Divergence = 0.5*KL(T||M) + 0.5*KL(S||M)
            # where M = 0.5*(T+S) — symmetric, bounded [0,1], stable
            gen_len = min(t_logits.shape[0], s_logits.shape[0])
            if gen_len == 0:
                del t_logits, s_logits
                bar.update(1); continue

            t_log = F.log_softmax(t_logits[:gen_len], dim=-1)  # teacher log-probs
            s_log = F.log_softmax(s_logits[:gen_len], dim=-1)  # student log-probs
            del t_logits, s_logits

            t_prob = t_log.exp()
            s_prob = s_log.exp()
            m_prob = 0.5 * (t_prob + s_prob)                   # mixture
            m_log  = m_prob.clamp(min=1e-10).log()

            # non-pad mask
            non_pad = (gen_only_gpu[0, :gen_len] != tokenizer.pad_token_id)
            del gen_only_gpu

            if non_pad.sum() == 0:
                del t_log, s_log, t_prob, s_prob, m_prob, m_log, non_pad
                bar.update(1); continue

            kl_t = F.kl_div(m_log, t_prob, reduction='none').sum(-1)  # [gen_len]
            kl_s = F.kl_div(m_log, s_prob, reduction='none').sum(-1)  # [gen_len]
            jsd  = (0.5 * kl_t + 0.5 * kl_s)[non_pad].mean()
            del t_log, s_log, t_prob, s_prob, m_prob, m_log, kl_t, kl_s, non_pad

            # ── Step 6: backward ─────────────────────────────────────────
            scaled = jsd / grad_accum
            scaled.backward()
            losses.append(scaled.item() * grad_accum)
            del scaled, jsd
            gc.collect(); torch.cuda.empty_cache()

            if (step+1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()), 1.0)
                optimizer.step(); optimizer.zero_grad()
                gc.collect(); torch.cuda.empty_cache()

            step_t = time.time()-t0
            times.append(step_t)
            avg_t  = sum(times[-10:])/len(times[-10:])
            done   = epoch*len(epoch_data)+step+1
            eta    = avg_t*(total_batches-done)
            vram   = torch.cuda.memory_allocated()/1e9
            tps    = total_toks/max(time.time()-train_start, 1e-6)

            bar.update(1)
            bar.set_postfix({'loss':f'{losses[-1]:.4f}',
                             'tok/s':f'{tps:.0f}',
                             'ETA':fmt_time(eta),
                             'VRAM':f'{vram:.1f}GB'})
            print(f'  step {done:4d}/{total_batches} loss={losses[-1]:.4f} '
                  f'tok/s={tps:.0f} '
                  f'elapsed={fmt_time(time.time()-train_start)} '
                  f'ETA={fmt_time(eta)} VRAM={vram:.1f}GB', flush=True)

    bar.close()
    print(f'  Total tokens: {total_toks:,}')
    return losses


print('Starting JSD training...')
t0 = time.time()
jsd_losses = opsd_train(student_model, data)
print(f'\n✅ JSD done in {fmt_time(time.time()-t0)} | loss={jsd_losses[-1]:.4f}')
student_model.save_pretrained(str(JSD_CKPT))
tokenizer.save_pretrained(str(JSD_CKPT))
import json as _j; (CKPT_DIR/'jsd_losses.json').write_text(_j.dumps(jsd_losses))
print(f'✅ Saved → {JSD_CKPT}')

In [ ]:
import json as _j

# Emergency partial save — captures whatever trained before any interrupt
student_model.save_pretrained(str(CKPT_DIR / 'jsd_adapter_partial'))
tokenizer.save_pretrained(str(CKPT_DIR / 'jsd_adapter_partial'))
try:
    (CKPT_DIR / 'jsd_losses_partial.json').write_text(_j.dumps(jsd_losses))
    print('✅ Partial JSD checkpoint + losses saved to jsd_adapter_partial')
except NameError:
    print('⚠️  jsd_losses not defined — model weights saved, no loss history available')


In [ ]:
# Inspect JSD adapter checkpoint files
jsd_path = CKPT_DIR / 'jsd_adapter'
if jsd_path.exists():
    for f in jsd_path.iterdir():
        print(f.name, f.stat().st_mtime)
else:
    print('❌ jsd_adapter not found — run Cell 07 first')


## 07b · Load JSD Checkpoint *(skip 07 if already trained)*

In [ ]:
import json as _j
from peft import PeftModel

if not JSD_CKPT.exists():
    print('❌ No JSD checkpoint — run Cell 07 first')
else:
    t0 = time.time()
    student_model = PeftModel.from_pretrained(base_model, str(JSD_CKPT))
    student_model.train(); student_model.enable_input_require_grads()
    jsd_losses = _j.loads((CKPT_DIR/'jsd_losses.json').read_text()) \
                 if (CKPT_DIR/'jsd_losses.json').exists() else []
    if jsd_losses:
        print(f'✅ JSD loaded in {time.time()-t0:.1f}s | {len(jsd_losses)} steps | loss={jsd_losses[-1]:.4f}')
    else:
        print(f'✅ JSD loaded in {time.time()-t0:.1f}s')


## 08 · Evaluation

In [ ]:
# ── Cell 08 · Evaluation — all checkpoints, saves results ───────────────────
import re as _re, json as _j
from pathlib import Path

EVAL_RESULTS_FILE = CKPT_DIR / 'eval_results_v14.json'

def extract_answer(text):
    if '####' in text: return text.split('####')[-1].strip()
    nums = _re.findall(r'-?\d+\.?\d*', text)
    return nums[-1] if nums else ''

def compute_reward(pred, y_star):
    gt = extract_answer(y_star)
    return int(bool(gt) and gt in pred)

def tokenize_batch_eval(prompts, max_len=180):
    return tokenizer(
        prompts, return_tensors='pt', padding=True,
        truncation=True, max_length=max_len
    ).to(DEVICE)

def evaluate_one(model, data, label, batch_size=BATCH_SIZE, n=100):
    model.eval()
    data_sub = data[:n]
    batches  = [data_sub[i:i+batch_size] for i in range(0, len(data_sub), batch_size)]
    correct  = 0
    bar = tqdm_nb(total=len(data_sub), desc=f'Eval [{label}]')
    for batch in batches:
        xs = [x for x,_ in batch]; ys = [y for _,y in batch]
        enc = tokenize_batch_eval(xs)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=200, do_sample=False,
                                  use_cache=True, pad_token_id=tokenizer.pad_token_id)
        texts = tokenizer.batch_decode(gen, skip_special_tokens=True)
        for txt, y in zip(texts, ys):
            correct += compute_reward(txt, y)
        bar.update(len(batch))
        bar.set_postfix({'acc': f'{correct/bar.n*100:.1f}%'})
    bar.close()
    return correct / len(data_sub) * 100

def print_results_table(results):
    print('\n' + '=' * 60)
    print('  EVALUATION SUMMARY')
    print('=' * 60)
    best = max(results, key=lambda x: x['accuracy'])
    for entry in results:
        filled = int(entry['accuracy'] / 2)
        bar    = '█' * filled + '░' * (50 - filled)
        crown  = ' 👑' if entry['label'] == best['label'] else ''
        print(f"  {entry['label']:16s} │ {bar} │ {entry['accuracy']:.1f}%{crown}")
    print('=' * 60)

def run_evaluation():
    from peft import PeftModel
    results = []
    # FIX: correct checkpoint list — JSD replaces the mislabeled GRPO entry
    checkpoints = [
        ('SFT',      SFT_CKPT),
        ('OPSD-abs', OPSD_ABS_CKPT),
        ('OPSD-KL',  OPSD_KL_CKPT),
        ('JSD',      JSD_CKPT),
    ]
    # Only include GRPO if a real GRPO checkpoint exists from a previous run
    if GRPO_CKPT.exists():
        checkpoints.append(('GRPO', GRPO_CKPT))
    for label, ckpt_path in checkpoints:
        if not ckpt_path.exists():
            print(f'  ⚠️  {label} checkpoint not found — skipping')
            continue
        print(f'\n── Evaluating {label} ──────────────────────────────────')
        gc.collect(); torch.cuda.empty_cache()
        eval_model = PeftModel.from_pretrained(base_model, str(ckpt_path))
        accuracy = evaluate_one(eval_model, data, label)
        results.append({'label': label, 'checkpoint': str(ckpt_path),
                        'accuracy': accuracy})
        print(f'  ✅ {label}: {accuracy:.1f}%')
        del eval_model
        gc.collect(); torch.cuda.empty_cache()
    print_results_table(results)
    EVAL_RESULTS_FILE.write_text(_j.dumps(results, indent=2))
    print(f'\n✅ Results saved → {EVAL_RESULTS_FILE}')
    return results

# ── Main: check saved, ask user ─────────────────────────────────────────────
if EVAL_RESULTS_FILE.exists():
    saved = _j.loads(EVAL_RESULTS_FILE.read_text())
    print('=' * 60)
    print('  ✅ SAVED EVALUATION RESULTS FOUND')
    print('=' * 60)
    print_results_table(saved)
    print(f'\n  File: {EVAL_RESULTS_FILE}')
    print('=' * 60)
    print('\nWould you like to re-evaluate all checkpoints?')
    print('  Type  yes  to re-run (~10-15 min per checkpoint)')
    print('  Press Enter  to keep saved results')
    user_input = input('  → ').strip().lower()
    if user_input in ('yes', 'y'):
        print('\n🔄 Re-running evaluation...')
        results = run_evaluation()
    else:
        print('\n✅ Using saved results.')
        results = saved
else:
    print('No saved results — running evaluation on all checkpoints...')
    results = run_evaluation()

acc = results[-1]['accuracy'] if results else 0.0
print(f'\n  acc variable = {acc:.1f}%  (used by Cell 09 Summary)')


## 09 · Results

In [ ]:
import json as _j
for var, fname in [('sft_losses',      'sft_losses.json'),
                   ('opsd_kl_losses',  'opsd_kl_losses.json'),
                   ('opsd_abs_losses', 'opsd_losses.json'),
                   ('jsd_losses',      'jsd_losses.json'),
                   ('grpo_losses',     'grpo_losses.json')]:
    if var not in globals() or not globals()[var]:
        f = CKPT_DIR/fname
        if f.exists(): globals()[var] = _j.loads(f.read_text())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('OPSD-KL vs JSD vs Baselines', fontsize=13, fontweight='bold')

ax = axes[0]
for ls, lbl, col in [
    (globals().get('sft_losses'),      'SFT',      '#4878CF'),
    (globals().get('opsd_abs_losses'), 'OPSD-abs', '#999999'),
    (globals().get('opsd_kl_losses'),  'OPSD-KL',  '#E8762D'),
    (globals().get('jsd_losses'),      'JSD',      '#6ABF6A'),
    (globals().get('grpo_losses'),     'GRPO',     '#9B59B6'),
]:
    if ls: ax.plot(ls, label=lbl, color=col, alpha=0.85, linewidth=1.5)
ax.set_title('Training Loss'); ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
if 'results' in globals() and results:
    labels = [r['label'] for r in results]
    accs   = [r['accuracy'] for r in results]
    color_map = {'SFT': '#4878CF', 'OPSD-abs': '#999999',
                 'OPSD-KL': '#E8762D', 'JSD': '#6ABF6A', 'GRPO': '#9B59B6'}
    colors = [color_map.get(l, '#AAAAAA') for l in labels]
    bars   = ax2.bar(labels, accs, color=colors, alpha=0.8, width=0.5)
    for bar, acc_val in zip(bars, accs):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{acc_val:.1f}%', ha='center', va='bottom', fontsize=9)
ax2.set_title('Accuracy Comparison'); ax2.set_ylabel('Accuracy %')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('opsd_v14_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: opsd_v14_results.png')


## 10 · Summary

In [ ]:
print('=' * 60)
print('  OPSD v14 EXPERIMENT SUMMARY')
print('=' * 60)
print(f'  GPU      : {torch.cuda.get_device_name(0)}')
print(f'  VRAM pk  : {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
print(f'  Dataset  : {DATASET_SIZE} samples')
print(f'  Batch    : {BATCH_SIZE}')
print()
print('  Loss curves:')
for name, var in [('SFT',      'sft_losses'),
                  ('OPSD-abs', 'opsd_abs_losses'),
                  ('OPSD-KL',  'opsd_kl_losses'),
                  ('JSD',      'jsd_losses'),
                  ('GRPO',     'grpo_losses')]:
    ls = globals().get(var)
    if ls: print(f'    {name:12s}: {len(ls)} steps, final loss = {ls[-1]:.4f}')
print()
print('  Accuracy:')
if 'results' in globals():
    for r in results:
        print(f"    {r['label']:16s}: {r['accuracy']:.1f}%")
print('=' * 60)
print('  Checkpoints:')
for name, path in [('SFT',      SFT_CKPT),
                   ('OPSD-abs', OPSD_ABS_CKPT),
                   ('OPSD-KL',  OPSD_KL_CKPT),
                   ('JSD',      JSD_CKPT),
                   ('GRPO',     GRPO_CKPT)]:
    print(f"    {'✅' if path.exists() else '❌'} {name:16s}: {path}")
print('=' * 60)


---
## Troubleshooting

### Flow for this notebook
```
Cell 01 → 02 → 03 → 04 → 05 (loads SFT) → 06 (KL-OPSD) → 07 (Eval)
```

### Cell 06 already trained — skip it
If `opsd_kl_adapter` exists, Cell 06 auto-skips. Just run it and proceed.

### OOM error
In Cell 02, reduce `BATCH_SIZE`:
```python
BATCH_SIZE = 2   # use if OOM with 4
BATCH_SIZE = 1   # use if still OOM
```

### teacher_model missing after kernel restart
```python
import copy
teacher_model = copy.deepcopy(student_model)
teacher_model.eval()
for p in teacher_model.parameters(): p.requires_grad = False
print(f'Teacher rebuilt | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
```

### What to watch during training
- `gap` column = `teacher_logprob - student_logprob`
- Gap shrinking toward 0 = student aligning with teacher
- Gap staying large = no learning
- `loss` going negative is normal — KL loss can be negative when student > teacher

### Comparing old vs new OPSD
- Old: `checkpoints/opsd_adapter` (abs loss)
- New: `checkpoints/opsd_kl_adapter` (KL loss)
- Both evaluated in Cell 07
